## Cell 1 — Install Required Libraries

In [ ]:
# Cell 1 — Install Required Libraries (Fixed v2)
!pip install -q "sympy>=1.13.3"
!pip install -q transformers torch sentence-transformers
!pip install -q bitsandbytes accelerate
!pip install -q pandas numpy scikit-learn

print("✅ Libraries installed!")

## Cell 2 — HuggingFace Login

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')  # Colab secrets se uthayega
login(token=hf_token)

print("✅ Logged in securely!")

## Cell 3 — Mount Google Drive

In [ ]:
# Cell 3 — Smart Path Detection (Colab + Local PC dono pe kaam karta hai)
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    # ── Colab mode ──
    DRIVE_PATH = "/content/drive/MyDrive/Dataset/"
    EMBED_PATH = "/content/drive/MyDrive/SLM_Embeddings/"
    IS_COLAB = True
    print("Running on Google Colab")
except:
    # ── Local PC mode ──
    BASE_DIR = os.getcwd()
    DRIVE_PATH = os.path.join(BASE_DIR, "data") + "/"
    EMBED_PATH = os.path.join(BASE_DIR, "embeddings") + "/"
    IS_COLAB = False
    os.makedirs(DRIVE_PATH, exist_ok=True)
    os.makedirs(EMBED_PATH, exist_ok=True)
    print("Running on Local PC")

print(f"   Data path:  {DRIVE_PATH}")
print(f"   Embed path: {EMBED_PATH}")
print(f"   Colab: {IS_COLAB}")


## Cell 4 — Load Outlet Dataset + 1000 Real Press Releases

In [ ]:
# Cell 4 — Load Outlet Dataset + 1000 Real Press Releases
import pandas as pd
import numpy as np

# ── Part 1: Outlet dataset (GitHub se) ──
outlet_url = "https://raw.githubusercontent.com/Tauqeerahmed1/MS-Thesis-v2/main/data/outlets_1000.csv"
df = pd.read_csv(outlet_url)

df['profile_text'] = (
    df['Media Outlet'].fillna('') + ' ' +
    df['Publication URL'].fillna('').str.replace('https://','').str.replace('http://','') + ' ' +
    df['Region'].fillna('')
).str.strip().str.lower()

print(f"✅ Outlet dataset loaded: {len(df)} outlets")

# ── Part 2: Real press releases (Drive se — already saved) ──
df_pr_full = pd.read_csv(DRIVE_PATH + 'final_combined_press_releases.csv')
print(f"✅ Full PR dataset loaded: {len(df_pr_full)} press releases")

# ── Part 3: Check karo — agar saved sample hai toh wahi load karo ──
import os
sample_path = DRIVE_PATH + 'sampled_1000_press_releases.csv'

if os.path.exists(sample_path):
    df_pr_sample = pd.read_csv(sample_path)
    print(f"✅ Loaded existing 1000 sample from Drive (same sample as before)")
else:
    df_pr_sample = df_pr_full.sample(n=1000, random_state=42).reset_index(drop=True)
    df_pr_sample.to_csv(sample_path, index=False)
    print(f"✅ New 1000 sample created and saved to Drive")

# ── Part 4: pr_texts banao ──
def make_pr_text(row):
    title = str(row['title'])
    content = str(row['content'])[:500]
    return f"{title}. {content}"

df_pr_sample['pr_text'] = df_pr_sample.apply(make_pr_text, axis=1)
pr_texts = df_pr_sample['pr_text'].tolist()

print(f"\n✅ {len(pr_texts)} real PR texts ready!")
print(f"\nSample (first 2):")
for i, t in enumerate(pr_texts[:2], 1):
    print(f"\n{i}. {t[:150]}...")

## Cell 5 — Generate Embeddings (All 5 SLM Models)

In [ ]:
# Cell 5 — Generate Embeddings (All 5 SLM Models)
# NOTE: Yeh cell ~1.5-2 ghante lega. Colab tab khula rakho.

import torch
import numpy as np
import time
import gc
import os
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device: {device}")

os.makedirs('/content/embeddings', exist_ok=True)

models_config = {
    "SmolLM":  "HuggingFaceTB/SmolLM2-135M",
    "Qwen":    "Qwen/Qwen2-1.5B",
    "Phi":     "microsoft/phi-2",
    "Mistral": "mistralai/Mistral-7B-v0.1",
    "Llama":   "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
}

def get_embeddings(texts, tokenizer, model, batch_size=16):
    all_embeddings = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt"
            ).to(device)
            outputs = model(**encoded)
            embeddings = outputs.last_hidden_state.mean(dim=1)
            embeddings = embeddings.to(torch.float32)
            embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
            all_embeddings.append(embeddings.cpu().numpy())
    return np.vstack(all_embeddings)

all_embeddings = {}
timing_results = {}

for model_name, model_id in models_config.items():
    print(f"\n{'='*50}")
    print(f"🔄 Starting {model_name} ({model_id})...")
    print(f"{'='*50}")

    # Skip karo agar already saved hai
    pr_file = f'/content/embeddings/{model_name}_pr.npy'
    outlet_file = f'/content/embeddings/{model_name}_outlet.npy'
    if os.path.exists(pr_file) and os.path.exists(outlet_file):
        print(f"⏭️  {model_name} already done — skip kar rahe hain")
        all_embeddings[model_name] = {
            "pr": np.load(pr_file),
            "outlet": np.load(outlet_file)
        }
        timing_results[model_name] = "already_done"
        continue

    try:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )

        use_4bit = model_name in ["Qwen", "Phi", "Mistral", "Llama"]

        tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        if use_4bit:
            model = AutoModel.from_pretrained(
                model_id,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True,
                torch_dtype=torch.float16
            )
        else:
            model = AutoModel.from_pretrained(
                model_id,
                trust_remote_code=True
            ).to(device)

        start = time.time()
        pr_emb = get_embeddings(pr_texts, tokenizer, model)
        outlet_emb = get_embeddings(df['profile_text'].tolist(), tokenizer, model)
        elapsed = time.time() - start

        all_embeddings[model_name] = {"pr": pr_emb, "outlet": outlet_emb}
        timing_results[model_name] = round(elapsed, 2)

        # Turant save karo
        np.save(pr_file, pr_emb)
        np.save(outlet_file, outlet_emb)

        print(f"✅ {model_name} done in {elapsed:.1f}s")
        print(f"   PR embeddings: {pr_emb.shape}")
        print(f"   Outlet embeddings: {outlet_emb.shape}")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    except Exception as e:
        print(f"❌ {model_name} failed: {e}")
        timing_results[model_name] = -1

print(f"\n{'='*50}")
print("✅ ALL MODELS DONE!")
print(f"{'='*50}")
for name, t in timing_results.items():
    print(f"  {name}: {t}s" if t != -1 else f"  {name}: FAILED")

## Cell 6 — Save Embeddings to Google Drive (Permanent)

In [ ]:
# Cell 6 — Save Embeddings to Google Drive (Permanent)
import shutil
import json
import os

save_path = '/content/drive/MyDrive/SLM_Embeddings'
os.makedirs(save_path, exist_ok=True)

shutil.copytree('/content/embeddings', save_path, dirs_exist_ok=True)

import json
with open(f'{save_path}/timing_results.json', 'w') as f:
    json.dump(timing_results, f, indent=2)

print("✅ All embeddings saved to Google Drive!")
print(f"\nLocation: {save_path}")
print(f"\nTiming Summary:")
for model, t in timing_results.items():
    status = f"{t}s" if t not in [-1, 'already_done'] else str(t)
    print(f"  {model}: {status}")

## Cell 7 — Load Embeddings from Drive (Session Restart ke baad yahan se shuru karo)

In [ ]:
# Cell 7 — Load Embeddings from Drive
# Agar session restart ho jaye toh Cells 1-4 + yeh cell run karo
import numpy as np
import os

EMBED_PATH = "/content/drive/MyDrive/SLM_Embeddings/"
model_names = ["SmolLM", "Qwen", "Phi", "Mistral", "Llama"]

pr_embeddings = {}
outlet_embeddings = {}

print("Loading embeddings from Google Drive...\n")

for model in model_names:
    pr_file = os.path.join(EMBED_PATH, f"{model}_pr.npy")
    outlet_file = os.path.join(EMBED_PATH, f"{model}_outlet.npy")

    if os.path.exists(pr_file) and os.path.exists(outlet_file):
        pr_embeddings[model] = np.load(pr_file)
        outlet_embeddings[model] = np.load(outlet_file)
        print(f"✅ {model} — PR: {pr_embeddings[model].shape}, Outlet: {outlet_embeddings[model].shape}")
    else:
        print(f"❌ {model} — Files not found! Pehle Cell 5 run karo.")

print("\n✅ Done loading embeddings!")

## Cell 8 — Cosine Similarity (PR vs Outlets)

In [ ]:
# Cell 8 — Cosine Similarity (PR vs Outlets)
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model_names = ["SmolLM", "Qwen", "Phi", "Mistral", "Llama"]
similarity_scores = {}

print("Computing cosine similarity for each model...\n")

for model in model_names:
    pr_emb = pr_embeddings[model]        # shape: (1000, dim)
    outlet_emb = outlet_embeddings[model]  # shape: (1000, dim)

    # Result: (1000 PRs x 1000 Outlets)
    scores = cosine_similarity(pr_emb, outlet_emb)
    similarity_scores[model] = scores

    print(f"✅ {model} — Similarity matrix: {scores.shape}")
    print(f"   Min: {scores.min():.4f} | Max: {scores.max():.4f} | Mean: {scores.mean():.4f}\n")

print("✅ Cosine similarity done for all models!")

In [ ]:
# Cell 8.5 — TF-IDF Baseline (Official baseline per thesis proposal)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Computing TF-IDF baseline...")

tfidf = TfidfVectorizer(max_features=1000, stop_words='english')

all_texts = pr_texts + df['profile_text'].tolist()
tfidf.fit(all_texts)

pr_tfidf = tfidf.transform(pr_texts).toarray()
outlet_tfidf = tfidf.transform(df['profile_text'].tolist()).toarray()

tfidf_scores = cosine_similarity(pr_tfidf, outlet_tfidf)

print(f"✅ TF-IDF baseline ready — shape: {tfidf_scores.shape}")
print(f"   Min: {tfidf_scores.min():.4f} | Max: {tfidf_scores.max():.4f} | Mean: {tfidf_scores.mean():.4f}")

## Cell 9 — Ranking Metrics (Precision@K, Recall@K, NDCG@K, MAP@K)

In [ ]:
# Cell 9 — Ranking Metrics (TF-IDF as neutral baseline reference)
import numpy as np

model_names = ["SmolLM", "Qwen", "Phi", "Mistral", "Llama"]
K_VALUES = [5, 10, 20]

def get_top_k_indices(scores, k):
    return np.argsort(scores, axis=1)[:, ::-1][:, :k]

def precision_at_k(predicted, ground_truth, k):
    scores = []
    for i in range(len(predicted)):
        pred_k = set(predicted[i][:k])
        gt_set = set(ground_truth[i])
        scores.append(len(pred_k & gt_set) / k)
    return np.mean(scores)

def recall_at_k(predicted, ground_truth, k):
    scores = []
    for i in range(len(predicted)):
        pred_k = set(predicted[i][:k])
        gt_set = set(ground_truth[i])
        scores.append(len(pred_k & gt_set) / len(gt_set))
    return np.mean(scores)

def ndcg_at_k(predicted, ground_truth, k):
    scores = []
    for i in range(len(predicted)):
        pred_k = predicted[i][:k]
        gt_set = set(ground_truth[i])
        dcg  = sum([1/np.log2(j+2) for j, idx in enumerate(pred_k) if idx in gt_set])
        idcg = sum([1/np.log2(j+2) for j in range(min(k, len(gt_set)))])
        scores.append(dcg/idcg if idcg > 0 else 0)
    return np.mean(scores)

def map_at_k(predicted, ground_truth, k):
    scores = []
    for i in range(len(predicted)):
        pred_k = predicted[i][:k]
        gt_set = set(ground_truth[i])
        hits, p_sum = 0, 0
        for j, idx in enumerate(pred_k):
            if idx in gt_set:
                hits += 1
                p_sum += hits / (j+1)
        scores.append(p_sum / min(k, len(gt_set)) if gt_set else 0)
    return np.mean(scores)

# ── TF-IDF ko reference banaya (official baseline per thesis proposal) ──
# Pehle Qwen self-reference tha jo biased tha — ab neutral, non-SLM baseline hai
gt_top20 = get_top_k_indices(tfidf_scores, 20)

# ── Compute & Print ──
print("=" * 65)
print(f"{'Model':<10} {'K':<5} {'P@K':<8} {'R@K':<8} {'NDCG@K':<10} {'MAP@K':<8}")
print("=" * 65)

results = {}
for model in model_names:
    pred_top20 = get_top_k_indices(similarity_scores[model], 20)
    results[model] = {}
    for k in K_VALUES:
        p = precision_at_k(pred_top20, gt_top20, k)
        r = recall_at_k(pred_top20, gt_top20, k)
        n = ndcg_at_k(pred_top20, gt_top20, k)
        m = map_at_k(pred_top20, gt_top20, k)
        results[model][k] = {"P@K": p, "R@K": r, "NDCG@K": n, "MAP@K": m}
        print(f"{model:<10} {k:<5} {p:<8.4f} {r:<8.4f} {n:<10.4f} {m:<8.4f}")
    print("-" * 65)

print("\n✅ Ranking metrics done! (TF-IDF used as neutral baseline reference)")

## Cell 10 — Visualizations (Speed + Quality Charts)

In [ ]:
# Cell 10 — Visualizations
import matplotlib.pyplot as plt
import numpy as np
import json, os

model_names = ["SmolLM", "Qwen", "Phi", "Mistral", "Llama"]

# ── Timing load karo ──
timing_file = '/content/drive/MyDrive/SLM_Embeddings/timing_results.json'
if os.path.exists(timing_file):
    with open(timing_file) as f:
        timing_results = json.load(f)
else:
    # Fallback — purane approximate values
    timing_results = {"SmolLM": 14.7, "Qwen": 6.1, "Phi": 7.2, "Mistral": 29.4, "Llama": 4.0}

timing_vals = [timing_results.get(m, 0) for m in model_names]
timing_vals = [t if isinstance(t, (int, float)) and t > 0 else 0 for t in timing_vals]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("SLM Encoder Benchmark — PR Distribution", fontsize=14, fontweight='bold')

# Plot 1: Speed Comparison
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']
axes[0].bar(model_names, timing_vals, color=colors)
axes[0].set_title("Embedding Time (seconds)")
axes[0].set_ylabel("Time (s)")
axes[0].set_xlabel("Model")
for i, v in enumerate(timing_vals):
    axes[0].text(i, v + 0.3, f"{v:.1f}s", ha='center', fontsize=9)

# Plot 2: NDCG@K comparison
k_vals = [5, 10, 20]
for model in model_names:
    ndcg_scores = [results[model][k]['NDCG@K'] for k in k_vals]
    axes[1].plot(k_vals, ndcg_scores, marker='o', label=model)
axes[1].set_title("NDCG@K Comparison")
axes[1].set_xlabel("K")
axes[1].set_ylabel("NDCG@K")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: MAP@K comparison
for model in model_names:
    map_scores = [results[model][k]['MAP@K'] for k in k_vals]
    axes[2].plot(k_vals, map_scores, marker='s', label=model)
axes[2].set_title("MAP@K Comparison")
axes[2].set_xlabel("K")
axes[2].set_ylabel("MAP@K")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/SLM_Embeddings/benchmark_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Charts saved to Drive!")

In [ ]:
# Cell 11 — Build CTR Proxy Labels (Similarity + Estimated Clicks)
import numpy as np
import pandas as pd

print("Building CTR proxy labels...")
print("Formula: 70% SLM similarity + 30% normalized Estimated Clicks\n")

# ── Step 1: Best SLM model ka similarity matrix lo (Qwen — best performer Stage 1 se) ──
BEST_MODEL = "Qwen"
sim_matrix = similarity_scores[BEST_MODEL]   # shape: (1000 PRs, 1000 Outlets)

print(f"✅ Using '{BEST_MODEL}' similarity matrix: {sim_matrix.shape}")

# ── Step 2: Similarity ko 0-1 normalize karo (min-max scaling) ──
sim_min, sim_max = sim_matrix.min(), sim_matrix.max()
sim_normalized = (sim_matrix - sim_min) / (sim_max - sim_min + 1e-9)

print(f"✅ Similarity normalized — range: [{sim_normalized.min():.4f}, {sim_normalized.max():.4f}]")

# ── Step 3: Estimated Clicks ko 0-1 normalize karo ──
clicks = df['Estimated Clicks'].values.astype(float)
clicks_min, clicks_max = clicks.min(), clicks.max()
clicks_normalized = (clicks - clicks_min) / (clicks_max - clicks_min + 1e-9)

print(f"✅ Clicks normalized — original range: [{clicks.min():.0f}, {clicks.max():.0f}]")
print(f"   Normalized range: [{clicks_normalized.min():.4f}, {clicks_normalized.max():.4f}]")

# ── Step 4: Combine — 70% similarity + 30% clicks ──
# clicks_normalized shape (1000,) → outlet ke liye hai, har PR ke liye same outlet-click value use hogi
# Broadcast karo: (1000 PRs, 1000 Outlets) ke har row mein clicks_normalized add karo

W_SIMILARITY = 0.7
W_CLICKS = 0.3

# clicks_normalized ko (1, 1000) shape do taake broadcast ho sake row-wise
clicks_broadcast = clicks_normalized.reshape(1, -1)  # shape: (1, 1000)

ctr_proxy_scores = (W_SIMILARITY * sim_normalized) + (W_CLICKS * clicks_broadcast)

print(f"\n✅ CTR Proxy Scores computed — shape: {ctr_proxy_scores.shape}")
print(f"   Min: {ctr_proxy_scores.min():.4f} | Max: {ctr_proxy_scores.max():.4f} | Mean: {ctr_proxy_scores.mean():.4f}")

# ── Sanity check — sample dekho ──
print(f"\n--- Sample: PR[0] ke top 3 outlets (by CTR proxy score) ---")
top3_idx = np.argsort(ctr_proxy_scores[0])[::-1][:3]
for idx in top3_idx:
    print(f"  {df.iloc[idx]['Media Outlet']:<25} | Score: {ctr_proxy_scores[0][idx]:.4f} "
          f"| Similarity: {sim_normalized[0][idx]:.4f} | Clicks(norm): {clicks_normalized[idx]:.4f}")

In [ ]:
# Cell 12 — Convert Continuous Score to Binary Label (Top-K threshold approach)
import numpy as np

print("Converting continuous CTR scores to binary labels...\n")

# ── Threshold approach: Top 20% pairs (per PR) ko "engaged" (1) maano ──
THRESHOLD_PERCENTILE = 80  # top 20%

binary_labels = np.zeros_like(ctr_proxy_scores)

for i in range(ctr_proxy_scores.shape[0]):
    threshold = np.percentile(ctr_proxy_scores[i], THRESHOLD_PERCENTILE)
    binary_labels[i] = (ctr_proxy_scores[i] >= threshold).astype(int)

print(f"✅ Binary labels created — shape: {binary_labels.shape}")
print(f"   Positive labels (1): {int(binary_labels.sum())} ({binary_labels.mean()*100:.1f}%)")
print(f"   Negative labels (0): {int((binary_labels==0).sum())} ({(1-binary_labels.mean())*100:.1f}%)")

# ── Sanity check ──
print(f"\n--- PR[0] ke labels (sample) ---")
positive_count = int(binary_labels[0].sum())
print(f"   Total positive outlets for PR[0]: {positive_count} / 1000")

In [ ]:
# Cell 13 — Build Feature Matrix for DeepFM (Full 1M pairs)
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

print("Building feature matrix for all 1,000,000 PR-Outlet pairs...\n")

n_prs = ctr_proxy_scores.shape[0]      # 1000
n_outlets = ctr_proxy_scores.shape[1]  # 1000

# ── Step 1: Outlet-level features encode karo (ek baar) ──
le_region = LabelEncoder()
le_outlet = LabelEncoder()

df['region_enc'] = le_region.fit_transform(df['Region'].fillna('Unknown'))
df['outlet_enc'] = le_outlet.fit_transform(df['Media Outlet'].fillna('Unknown'))

# Normalize traffic/views/clicks (0-1)
df['traffic_norm'] = (df['Estimated Traffic'] - df['Estimated Traffic'].min()) / \
                      (df['Estimated Traffic'].max() - df['Estimated Traffic'].min() + 1e-9)
df['views_norm'] = (df['Estimated Views'] - df['Estimated Views'].min()) / \
                    (df['Estimated Views'].max() - df['Estimated Views'].min() + 1e-9)
df['clicks_norm'] = clicks_normalized  # already computed Cell 11 mein

print(f"✅ Outlet features encoded — {len(df)} outlets")
print(f"   Unique regions: {df['region_enc'].nunique()}")
print(f"   Unique outlets: {df['outlet_enc'].nunique()}")

# ── Step 2: Saare PR-Outlet pairs ke liye rows banao (vectorized — fast) ──
pr_indices = np.repeat(np.arange(n_prs), n_outlets)      # [0,0,0,...,1,1,1,...]
outlet_indices = np.tile(np.arange(n_outlets), n_prs)    # [0,1,2,...,0,1,2,...]

print(f"\n✅ Generated {len(pr_indices):,} pair indices")

# ── Step 3: Feature matrix banao ──
feature_df = pd.DataFrame({
    'pr_idx': pr_indices,
    'outlet_idx': outlet_indices,
    'region_enc': df['region_enc'].values[outlet_indices],
    'outlet_enc': df['outlet_enc'].values[outlet_indices],
    'traffic_norm': df['traffic_norm'].values[outlet_indices],
    'views_norm': df['views_norm'].values[outlet_indices],
    'clicks_norm': df['clicks_norm'].values[outlet_indices],
    'similarity': sim_normalized[pr_indices, outlet_indices],
    'label': binary_labels[pr_indices, outlet_indices].astype(int)
})

print(f"\n{'='*50}")
print(f"✅ FEATURE MATRIX READY")
print(f"{'='*50}")
print(f"Total rows: {len(feature_df):,}")
print(f"Columns: {feature_df.columns.tolist()}")
print(f"\nLabel distribution:")
print(feature_df['label'].value_counts())
print(f"\nSample rows:")
print(feature_df.head(5))

# Save to Drive (taake dobara banana na pade)
feature_df.to_csv(DRIVE_PATH + 'deepfm_feature_matrix.csv', index=False)
print(f"\n✅ Saved to Drive: deepfm_feature_matrix.csv")
print(f"   File size estimate: ~{len(feature_df) * 9 * 8 / 1024 / 1024:.1f} MB")

In [ ]:
# Cell 14 — DeepFM Model (NumPy Implementation) + Train/Test Split
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
import numpy as np

print("Preparing data for DeepFM training...\n")

# ── Features select karo ──
feature_cols = ['region_enc', 'outlet_enc', 'traffic_norm',
                 'views_norm', 'clicks_norm', 'similarity']

X = feature_df[feature_cols].values.astype(np.float64)
y = feature_df['label'].values.astype(np.float64)

# ── Train/Test split (80/20, stratified taake class balance maintain rahe) ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✅ Train set: {X_train.shape[0]:,} rows")
print(f"✅ Test set:  {X_test.shape[0]:,} rows")
print(f"   Train positive ratio: {y_train.mean()*100:.1f}%")
print(f"   Test positive ratio:  {y_test.mean()*100:.1f}%")


# ════════════════════════════════════════════
# DeepFM — NumPy Implementation
# (FM layer: linear + pairwise interactions
#  Deep layer: simple feedforward NN)
# ════════════════════════════════════════════

class DeepFM_NumPy:
    def __init__(self, n_features, embed_dim=8, hidden_dims=[32, 16], lr=0.01, seed=42):
        np.random.seed(seed)
        self.n_features = n_features
        self.embed_dim = embed_dim
        self.lr = lr

        # FM part: linear weights + embeddings (for pairwise interaction)
        self.w0 = 0.0
        self.w1 = np.random.normal(0, 0.01, n_features)
        self.V = np.random.normal(0, 0.01, (n_features, embed_dim))  # embeddings for FM 2nd order

        # Deep part: simple 2-layer feedforward
        self.W1 = np.random.normal(0, 0.1, (n_features, hidden_dims[0]))
        self.b1 = np.zeros(hidden_dims[0])
        self.W2 = np.random.normal(0, 0.1, (hidden_dims[0], hidden_dims[1]))
        self.b2 = np.zeros(hidden_dims[1])
        self.W3 = np.random.normal(0, 0.1, (hidden_dims[1], 1))
        self.b3 = 0.0

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

    def relu(self, x):
        return np.maximum(0, x)

    def forward(self, X):
        # ── FM part ──
        linear_term = self.w0 + X @ self.w1
        # Pairwise interaction (standard FM trick): 0.5 * sum[(sum(vx))^2 - sum((vx)^2)]
        Vx = X @ self.V                      # (n, embed_dim)
        term1 = np.sum(Vx**2, axis=1)
        term2 = np.sum((X**2) @ (self.V**2), axis=1)
        fm_term = 0.5 * (term1 - term2)

        fm_output = linear_term + fm_term

        # ── Deep part ──
        h1 = self.relu(X @ self.W1 + self.b1)
        h2 = self.relu(h1 @ self.W2 + self.b2)
        deep_output = (h2 @ self.W3 + self.b3).flatten()

        # ── Combine ──
        logits = fm_output + deep_output
        return self.sigmoid(logits), h1, h2, Vx

    def train_batch(self, X_batch, y_batch):
        n = X_batch.shape[0]
        y_pred, h1, h2, Vx = self.forward(X_batch)

        # Loss gradient (binary cross-entropy derivative w.r.t logits)
        d_logits = (y_pred - y_batch) / n

        # ── Update FM part ──
        self.w0 -= self.lr * np.sum(d_logits)
        self.w1 -= self.lr * (X_batch.T @ d_logits)

        # V gradient (simplified)
        for f in range(self.embed_dim):
            grad_V = (X_batch.T @ (d_logits * Vx[:, f])) - (self.V[:, f] * np.sum((X_batch**2).T @ d_logits, axis=0).mean())
            self.V[:, f] -= self.lr * grad_V * 0.01  # scaled down for stability

        # ── Update Deep part (simple backprop) ──
        d_h2 = np.outer(d_logits, self.W3.flatten())
        d_h2[h2 <= 0] = 0
        self.W3 -= self.lr * (h2.T @ d_logits.reshape(-1,1))
        self.b3 -= self.lr * np.sum(d_logits)

        d_h1 = d_h2 @ self.W2.T
        d_h1[h1 <= 0] = 0
        self.W2 -= self.lr * (h1.T @ d_h2)
        self.b2 -= self.lr * np.sum(d_h2, axis=0)

        self.W1 -= self.lr * (X_batch.T @ d_h1)
        self.b1 -= self.lr * np.sum(d_h1, axis=0)

        # Loss compute karo (monitoring ke liye)
        eps = 1e-9
        loss = -np.mean(y_batch * np.log(y_pred + eps) + (1 - y_batch) * np.log(1 - y_pred + eps))
        return loss

    def predict(self, X):
        y_pred, _, _, _ = self.forward(X)
        return y_pred


print("\n✅ DeepFM class defined! Run next cell to train.")

In [ ]:
# Cell 15 — Train DeepFM (10 epochs, batch_size=1024)
import time

print("Starting DeepFM training...")
print(f"Train rows: {X_train.shape[0]:,} | Batch size: 1024 | Epochs: 10\n")

# ── Feature scaling (zaroori hai NumPy DeepFM ke liye — stability ke liye) ──
# region_enc aur outlet_enc large integers hain (0-51, 0-999) — inhe bhi 0-1 normalize karte hain
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

for col_idx in range(X_train.shape[1]):
    col_min = X_train[:, col_idx].min()
    col_max = X_train[:, col_idx].max()
    if col_max > col_min:
        X_train_scaled[:, col_idx] = (X_train[:, col_idx] - col_min) / (col_max - col_min)
        X_test_scaled[:, col_idx] = (X_test[:, col_idx] - col_min) / (col_max - col_min)

print("✅ Features scaled to 0-1 range\n")

# ── Model initialize karo ──
model = DeepFM_NumPy(n_features=X_train.shape[1], embed_dim=8, hidden_dims=[32, 16], lr=0.05)

BATCH_SIZE = 1024
EPOCHS = 10
n_train = X_train_scaled.shape[0]
n_batches = n_train // BATCH_SIZE

print(f"Batches per epoch: {n_batches}\n")
print("="*60)

start_time = time.time()
loss_history = []

for epoch in range(EPOCHS):
    epoch_start = time.time()

    # Shuffle data har epoch
    perm = np.random.permutation(n_train)
    X_shuffled = X_train_scaled[perm]
    y_shuffled = y_train[perm]

    epoch_losses = []
    for b in range(n_batches):
        start_idx = b * BATCH_SIZE
        end_idx = start_idx + BATCH_SIZE
        X_batch = X_shuffled[start_idx:end_idx]
        y_batch = y_shuffled[start_idx:end_idx]

        loss = model.train_batch(X_batch, y_batch)
        epoch_losses.append(loss)

    avg_loss = np.mean(epoch_losses)
    loss_history.append(avg_loss)
    epoch_time = time.time() - epoch_start

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | Loss: {avg_loss:.4f} | Time: {epoch_time:.1f}s")

total_time = time.time() - start_time
print("="*60)
print(f"\n✅ Training complete! Total time: {total_time/60:.1f} minutes")

In [ ]:
# Cell 16 — Evaluate DeepFM on Test Set
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

print("Evaluating on test set...\n")

y_pred_proba = model.predict(X_test_scaled)
y_pred_binary = (y_pred_proba >= 0.5).astype(int)

# ── Metrics ──
acc = accuracy_score(y_test, y_pred_binary)
auc = roc_auc_score(y_test, y_pred_proba)

print("="*60)
print(f"✅ DeepFM Test Results")
print("="*60)
print(f"Accuracy: {acc:.4f} ({acc*100:.2f}%)")
print(f"AUC-ROC:  {auc:.4f}")
print()

print("Classification Report:")
print(classification_report(y_test, y_pred_binary, target_names=['Not Engaged (0)', 'Engaged (1)']))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred_binary)
print(f"                 Predicted 0   Predicted 1")
print(f"Actual 0         {cm[0][0]:<13} {cm[0][1]}")
print(f"Actual 1         {cm[1][0]:<13} {cm[1][1]}")

# Prediction distribution check (sanity)
print(f"\nPrediction probability stats:")
print(f"  Min: {y_pred_proba.min():.4f} | Max: {y_pred_proba.max():.4f} | Mean: {y_pred_proba.mean():.4f}")

## Stage 3 — PageRank Authority Reranking

Build outlet-outlet co-mention graph using embedding similarity, compute PageRank, and combine into final hybrid score.

In [ ]:
# Cell 17 — Outlet-Outlet Similarity Matrix
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Load Qwen outlet embeddings
outlet_embeddings = np.load(EMBED_PATH + "Qwen_outlet.npy")
print(f"Outlet embeddings shape: {outlet_embeddings.shape}")

print("Computing outlet-outlet similarity matrix...")
outlet_sim_matrix = cosine_similarity(outlet_embeddings)
print(f"Similarity matrix shape: {outlet_sim_matrix.shape}")
print(f"Diagonal check (should be 1.0): {outlet_sim_matrix[0,0]:.4f}")
print(f"Sample row[0] first 5: {outlet_sim_matrix[0, :5]}")


In [ ]:
# Cell 18 — Build Co-mention Graph
import numpy as np

print("Building outlet-outlet co-mention graph...")
EDGE_THRESHOLD = 0.95
adj_matrix = (outlet_sim_matrix >= EDGE_THRESHOLD).astype(float)
np.fill_diagonal(adj_matrix, 0)

n_nodes = adj_matrix.shape[0]
n_edges = int(adj_matrix.sum() / 2)
print(f"Graph built:")
print(f"   Nodes (outlets): {n_nodes}")
print(f"   Edges (connections): {n_edges:,}")
print(f"   Threshold used: {EDGE_THRESHOLD}")
print(f"   Avg connections per outlet: {adj_matrix.sum(axis=1).mean():.1f}")
print(f"   Isolated nodes: {int((adj_matrix.sum(axis=1) == 0).sum())}")


In [ ]:
# Cell 19 — PageRank (Power Iteration)
import numpy as np

print("Computing PageRank on outlet graph...")
DAMPING = 0.85
MAX_ITER = 100
TOL = 1e-6
n = adj_matrix.shape[0]

col_sums = adj_matrix.sum(axis=0)
col_sums[col_sums == 0] = 1
transition_matrix = adj_matrix / col_sums

pr = np.ones(n) / n
for iteration in range(MAX_ITER):
    pr_new = (1 - DAMPING) / n + DAMPING * (transition_matrix @ pr)
    diff = np.abs(pr_new - pr).sum()
    pr = pr_new
    if diff < TOL:
        print(f"Converged at iteration {iteration + 1}")
        break

pr_normalized = (pr - pr.min()) / (pr.max() - pr.min() + 1e-9)
print(f"PageRank computed:")
print(f"   Min: {pr.min():.6f} | Max: {pr.max():.6f} | Mean: {pr.mean():.6f}")

top10_idx = np.argsort(pr)[::-1][:10]
print(f"Top 10 Outlets by PageRank Authority:")
for rank, idx in enumerate(top10_idx, 1):
    print(f"  {rank:2d}. {df.iloc[idx]['Media Outlet']:<30} | PR: {pr[idx]:.6f} | Norm: {pr_normalized[idx]:.4f}")


In [ ]:
# Cell 20 — Final Hybrid Score
import numpy as np

print("Computing Final Hybrid Recommendation Score...")
print("Formula: Score_i = 0.5*Relevance + 0.3*CTR + 0.2*PageRank")

LAMBDA_1 = 0.5
LAMBDA_2 = 0.3
LAMBDA_3 = 0.2

relevance = sim_normalized
ctr = (ctr_proxy_scores - ctr_proxy_scores.min()) / (ctr_proxy_scores.max() - ctr_proxy_scores.min() + 1e-9)
pagerank = pr_normalized.reshape(1, -1)

hybrid_score = LAMBDA_1 * relevance + LAMBDA_2 * ctr + LAMBDA_3 * pagerank

print(f"Hybrid score matrix: {hybrid_score.shape}")
print(f"   Min: {hybrid_score.min():.4f} | Max: {hybrid_score.max():.4f} | Mean: {hybrid_score.mean():.4f}")

print("\nSample Recommendations (Top 5 per PR)")
print("="*60)
for pr_i in range(3):
    top5_idx = np.argsort(hybrid_score[pr_i])[::-1][:5]
    pr_title = df_pr_sample.iloc[pr_i]['title'][:60]
    print(f"\nPR {pr_i+1}: '{pr_title}...'")
    print(f"{'Rank':<6}{'Outlet':<30}{'Hybrid':>8}{'Relevance':>11}{'CTR':>8}{'PageRank':>10}")
    print("-"*60)
    for rank, idx in enumerate(top5_idx, 1):
        print(f"{rank:<6}{df.iloc[idx]['Media Outlet']:<30}"
              f"{hybrid_score[pr_i,idx]:>8.4f}"
              f"{relevance[pr_i,idx]:>11.4f}"
              f"{ctr[pr_i,idx]:>8.4f}"
              f"{pr_normalized[idx]:>10.4f}")


In [ ]:
# Cell 21 — Save Stage 3 Results
import numpy as np, json

np.save(DRIVE_PATH + "pagerank_scores.npy", pr)
np.save(DRIVE_PATH + "pagerank_scores_normalized.npy", pr_normalized)
np.save(DRIVE_PATH + "hybrid_score_matrix.npy", hybrid_score)
np.save(DRIVE_PATH + "outlet_sim_matrix.npy", outlet_sim_matrix)

summary = {
    "stage3_pagerank": {
        "graph_nodes": int(n_nodes), "graph_edges": int(n_edges),
        "edge_threshold": EDGE_THRESHOLD, "damping_factor": DAMPING,
        "pr_min": float(pr.min()), "pr_max": float(pr.max()),
        "top_outlet": df.iloc[pr.argmax()]["Media Outlet"]
    },
    "final_hybrid": {
        "lambda1": LAMBDA_1, "lambda2": LAMBDA_2, "lambda3": LAMBDA_3,
        "score_min": float(hybrid_score.min()),
        "score_max": float(hybrid_score.max()),
        "matrix_shape": list(hybrid_score.shape)
    }
}
with open(DRIVE_PATH + "stage3_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Saved: pagerank_scores.npy, hybrid_score_matrix.npy, stage3_summary.json")
print("="*60)
print("  ALL 3 STAGES COMPLETE — THESIS PIPELINE DONE!")
print("="*60)
